In [47]:
import pandas as pd
import peptides

# Load the CSV files
df = pd.read_csv('dataset_80_20_normalized_tcr_with_cohort.csv')
df2 = pd.read_csv('prediction.csv')



In [48]:
df.head()

,Cancer_Type,gene,Peptide,wt_seq,pep_mut_start,mutant_best_alleles,mutant_rank,mutant_rank_netMHCpan,mutant_rank_PRIME,mut_Rank_Stab,...,immunogenicity,foreignness_score,hydrophobicity_avg_diff,hydrophobicity_total_diff,mut_Thalf,mw_tcr_contact,hydroph_tcr_contact,charge_tcr_contact,mutant_best_alleles_normalized,cohort
0,Melanoma,CHD5,KSKKEFRAY,KSEKEFRAY,3.0,HLA-C12:03,2.947,3.6904,2.947,0.8,...,0,0.483,-0.044,-0.4,1.6604,578.66894,-2.275,0.994865,C1203,HiTIDE
1,Melanoma,MYOCD,QSFSSTSAL,QSSSSTSAL,3.0,HLA-B07:02,1.779,1.8105,1.779,3.0,...,0,0.655,0.400,3.6,0.3099,380.35494,-0.775,-0.005094,B0702,NaN
2,Melanoma,GSE1,QAPDPAYIY,RAPDPAYIY,1.0,HLA-C07:02,0.039,0.0507,0.039,1.8,...,0,0.366,0.111,1.0,0.1162,464.47534,-1.150,-1.007049,C0702,NCI
3,Melanoma,SUPT16H,GSDAEEGDSA,GSDAEEGDSE,10.0,HLA-A01:01,4.563,8.1864,4.563,42.0,...,0,0.669,0.530,5.3,0.1289,519.46554,-1.820,-3.003501,A0101,HiTIDE
4,Melanoma,FGF7,WYLRINKRGKV,WYLRIDKRGKV,6.0,HLA-A24:02,22.955,0.5629,22.955,3.5,...,0,0.379,0.000,0.0,0.2800,742.87944,-2.050,2.994149,A2402,NaN


In [49]:
# En df2 me quedo con las combinaciones de Antigen HLA que tengan el mayor Rank.

df2 = df2.loc[df2.groupby(['Antigen', 'HLA'])['Rank'].idxmin()]
df2.reset_index(drop=True, inplace=True)

In [50]:
df2 

,CDR3,Antigen,HLA,Rank
0,CASSTMGNAPLDTQYF,AAAAAAQQIQV,B*07:02,0.0000
1,CASSLHLLRGLNTGELFF,AAAAAGVSFTK,A*11:01,0.0144
2,CASSQGGDYGYTF,AAAAGVSFTK,A*11:01,0.0178
3,CASGDSGDEQFF,AAAAQQIQV,A*02:01,0.0042
4,CAGGSYNNNDMRF,AAAEKLPSL,B*07:02,0.0020
...,...,...,...,...
8165,CAVGGQKLGDTQYF,YYPGKKEPF,A*24:02,0.0410
8166,CSARGLAEANTGELFF,YYQTETYDL,A*24:07,0.0078
8167,CAVGGQKLGDTQYF,YYSPQAQTL,A*24:02,0.0141
8168,CASSEDSGGYNEQFF,YYTNLEQKL,C*06:02,0.0021


In [51]:
#Todos los valores negativos en Rank los cambio a 0
df2['Rank'] = df2['Rank'].apply(lambda x: max(x, 0))

In [52]:
import re
# 1. Recuperamos la función de limpieza que usamos al principio
# Esto asegura que el formato sea IDÉNTICO en ambos lados (ej: C*12:03)
def corregir_hla(hla_string):
    if pd.isna(hla_string): return "Unknown"
    # Quitar HLA-, quedarse con el primero y poner el asterisco
    hla = str(hla_string).split(',')[0].strip()
    hla = hla.replace("HLA-", "").replace("hla-", "")
    return re.sub(r'([ABC])(\d)', r'\1*\2', hla)

# 2. Creamos una columna temporal en tu DF original para hacer el cruce
# Ahora df tendrá una columna 'hla_join' que será igualita a la de df2 ('C*12:03')
df['hla_join'] = df['mutant_best_alleles'].apply(corregir_hla)

# 3. Hacemos el MERGE (Cruce inteligente)
# Unimos df con df2 usando las columnas coincidentes
# left_on: columnas en df
# right_on: columnas en df2
df_final = df.merge(
    df2[['Antigen', 'HLA', 'Rank']], # Solo traemos lo que nos interesa de df2
    left_on=['Peptide', 'hla_join'], 
    right_on=['Antigen', 'HLA'], 
    how='left' # 'left' significa: quédate con todo lo de df y pega df2 donde coincida
)

# 4. Limpieza final (Opcional)
# Si quieres, borra las columnas auxiliares que se han duplicado
df_final.drop(columns=['hla_join', 'Antigen', 'HLA'], inplace=True)

# Verificación
print(df_final['Rank'].head())
print(f"Número de nulos en Rank: {df_final['Rank'].isna().sum()}")

0    0.0087
1    0.0036
2    0.0022
3    0.0155
4    0.0047
Name: Rank, dtype: float64
Número de nulos en Rank: 2613


In [54]:
df_final

,Cancer_Type,gene,Peptide,wt_seq,pep_mut_start,mutant_best_alleles,mutant_rank,mutant_rank_netMHCpan,mutant_rank_PRIME,mut_Rank_Stab,...,foreignness_score,hydrophobicity_avg_diff,hydrophobicity_total_diff,mut_Thalf,mw_tcr_contact,hydroph_tcr_contact,charge_tcr_contact,mutant_best_alleles_normalized,cohort,Rank
0,Melanoma,CHD5,KSKKEFRAY,KSEKEFRAY,3.0,HLA-C12:03,2.947,3.6904,2.947,0.80,...,0.483,-0.044,-0.4,1.6604,578.66894,-2.275,0.994865,C1203,HiTIDE,0.0087
1,Melanoma,MYOCD,QSFSSTSAL,QSSSSTSAL,3.0,HLA-B07:02,1.779,1.8105,1.779,3.00,...,0.655,0.400,3.6,0.3099,380.35494,-0.775,-0.005094,B0702,NaN,0.0036
2,Melanoma,GSE1,QAPDPAYIY,RAPDPAYIY,1.0,HLA-C07:02,0.039,0.0507,0.039,1.80,...,0.366,0.111,1.0,0.1162,464.47534,-1.150,-1.007049,C0702,NCI,0.0022
3,Melanoma,SUPT16H,GSDAEEGDSA,GSDAEEGDSE,10.0,HLA-A01:01,4.563,8.1864,4.563,42.00,...,0.669,0.530,5.3,0.1289,519.46554,-1.820,-3.003501,A0101,HiTIDE,0.0155
4,Melanoma,FGF7,WYLRINKRGKV,WYLRIDKRGKV,6.0,HLA-A24:02,22.955,0.5629,22.955,3.50,...,0.379,0.000,0.0,0.2800,742.87944,-2.050,2.994149,A2402,NaN,0.0047
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11183,Melanoma,USP34,NLFRRVWEL,NLSRRVWEL,3.0,HLA-B08:01,0.018,0.0165,0.018,0.30,...,0.657,0.400,3.6,0.3060,615.73604,-1.425,1.994890,B0801,NaN,0.0006
11184,Melanoma,USP34,NLFRRVWEL,NLSRRVWEL,3.0,HLA-B08:01,0.018,0.0165,0.018,0.30,...,0.657,0.400,3.6,0.3060,615.73604,-1.425,1.994890,B0801,NaN,0.0006
11185,Melanoma,FGGY,FLLWKATGV,FLSWKATGV,3.0,HLA-A02:01,0.626,0.0538,0.626,0.05,...,0.629,0.511,4.6,6.7224,504.58644,-0.925,0.994166,A0201,TESLA,0.0005
11186,Melanoma,FOXJ3,FSASFRSLY,LSASFRSLY,1.0,B1501,0.323,0.4161,0.323,1.10,...,0.282,-0.111,-1.0,0.4359,495.53574,-0.825,0.994898,B1501,NaN,NaN


In [57]:
#Compruebo cuantos valores nulos hay en la columna Rank
df_final['Rank'].isnull().sum()

#Creo un dataframe con los valores nulos que contiene el mutant_best_alleles, Peptide
df_nulos = df_final[df_final['Rank'].isnull()][['mutant_best_alleles', 'Peptide']]
df_nulos

,mutant_best_alleles,Peptide
6,B2705,GRWALVSPE
7,A3002,QSFEKQGFY
8,A0101,YTDFHCQYV
9,A0101,YTDFHCQYV
18,A2402,HYKDAHVDF
...,...,...
11174,B0801,SLKLHYQL
11175,A2402,IICSSVDIHF
11179,A0101,SYTTNAFSGEY
11180,A0101,SYTTNAFSGEY


In [58]:
#Guardo el dataframe de nulos en un csv
df_nulos.to_csv('df_nulos_tcr_try.csv', index=False)